# Qwen3-TTS · Audio italiani del prototipo NLS
Notebook autonomo: contiene i testi dei **24 audio** (18 passaggi, 3 supporti, 3 messaggi).

1. In Colab scegli **Runtime → Cambia tipo di runtime → GPU**. Una T4 è il punto di partenza; disponibilità e limiti dipendono da Colab.
2. Esegui installazione e configurazione, poi genera e ascolta i due campioni.
3. Scegli il campione nella cella «Selezione» e genera tutti gli audio.
4. Scarica lo ZIP prima che la sessione Colab termini.

Usa **float16 su T4** e SDPA, senza FlashAttention 2. VoiceDesign e Base vengono caricati uno alla volta. Se Base 1.7B esaurisce la memoria, seleziona Base 0.6B e riesegui dal caricamento Base. La compatibilità effettiva dipende dalle librerie e dalla GPU assegnata; questo notebook non è stato eseguito su Colab dall’autore.

Il modello legge testi predefiniti; non sceglie interventi. Ascolta e verifica testi, numeri e pronunce prima dell’integrazione. I WAV non vengono collegati automaticamente all’app.

Fonti: [Qwen3-TTS ufficiale](https://github.com/QwenLM/Qwen3-TTS), [modelli Qwen](https://huggingface.co/collections/Qwen/qwen3-tts). Pesi e codice Qwen: Apache 2.0. Nessuna chiave API richiesta; il primo avvio scarica alcuni GB di modelli.


## 1 · Installazione
Usa una sessione Colab nuova. Se Colab chiede un riavvio dopo l’installazione, riavvia e prosegui dalla configurazione.


In [ ]:
%pip install -q "qwen-tts==0.1.1" soundfile
!apt-get -qq update > /dev/null 2>&1
!apt-get -qq -y install sox libsox-fmt-all > /dev/null 2>&1


## 2 · Configurazione
La descrizione della voce si può modificare. I due campioni differiscono per il seme di generazione.


In [ ]:
from pathlib import Path
import gc, hashlib, importlib.metadata, json, re, shutil, zipfile
import numpy as np
import soundfile as sf
import torch
from IPython.display import Audio, display
from qwen_tts import Qwen3TTSModel

assert torch.cuda.is_available(), "Attiva una GPU da Runtime → Cambia tipo di runtime."
props = torch.cuda.get_device_properties(0)
print(f"GPU: {props.name} | memoria: {props.total_memory / 2**30:.1f} GiB")
DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
ATTENTION = "sdpa"
DESIGN_MODEL = "Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign"
BASE_MODEL = "Qwen/Qwen3-TTS-12Hz-1.7B-Base"  # alternativa: Qwen/Qwen3-TTS-12Hz-0.6B-Base
VOICE_DESCRIPTION = (
    "An adult female voice speaking native standard Italian, clear and natural, "
    "calm and professional. Moderate, unhurried pace with short pauses between sentences. "
    "Precise articulation of numbers and medical terminology. Neutral delivery, no theatrical emotion, "
    "no whispering. Clean studio recording without background sounds."
)
REFERENCE_TEXT = (
    "Valuta il respiro e rileva la frequenza cardiaca. "
    "Inserisci il valore in battiti al minuto, poi conferma la valutazione. "
    "Comunica al team gli interventi e la risposta del neonato."
)
CANDIDATE_SEEDS = [42, 123]
WORK = Path("/content/nls-qwen")
WORK.mkdir(parents=True, exist_ok=True)

def release_models():
    for name in ("design_model", "clone_model", "voice_prompt"):
        globals().pop(name, None)
    gc.collect()
    torch.cuda.empty_cache()

def load_model(model_id):
    return Qwen3TTSModel.from_pretrained(
        model_id, device_map="cuda:0", dtype=DTYPE, attn_implementation=ATTENTION)

def checked_audio(wave):
    wave = np.asarray(wave, dtype=np.float32).reshape(-1)
    if not wave.size or not np.isfinite(wave).all() or np.max(np.abs(wave)) < 1e-6:
        raise ValueError("Audio vuoto, silenzioso o non valido.")
    return wave

print("Precisione:", DTYPE, "| attenzione:", ATTENTION)


## 3 · Genera e ascolta i campioni
Il riferimento deve dire **esattamente** il testo mostrato: verifica che non ometta o aggiunga parole. Per altre prove cambia descrizione o semi, poi riesegui questa cella.


In [ ]:
release_models()
design_model = load_model(DESIGN_MODEL)
candidates = []
try:
    for number, seed in enumerate(CANDIDATE_SEEDS, 1):
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        with torch.inference_mode():
            waves, sr = design_model.generate_voice_design(
                text=REFERENCE_TEXT, language="Italian", instruct=VOICE_DESCRIPTION,
                max_new_tokens=2048, do_sample=True, temperature=0.7)
        wave = checked_audio(waves[0])
        path = WORK / f"reference-{number}.wav"
        sf.write(path, wave, int(sr), subtype="PCM_16")
        metadata = {"text": REFERENCE_TEXT, "description": VOICE_DESCRIPTION,
                    "seed": seed, "model": DESIGN_MODEL, "sample_rate": int(sr)}
        path.with_suffix(".json").write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")
        candidates.append(path)
        print(f"Campione {number}, seme {seed}: {len(wave)/sr:.1f} secondi")
        print(REFERENCE_TEXT)
        display(Audio(filename=str(path)))
finally:
    release_models()


## 4 · Selezione della voce
Dopo l’ascolto, imposta `SELECTED_CANDIDATE` a 1 oppure 2 e `REFERENCE_CHECKED` a `True`. Conserva lo ZIP finale: contiene anche il riferimento per riutilizzare la stessa voce in futuro.


In [ ]:
SELECTED_CANDIDATE = 1
REFERENCE_CHECKED = False  # True dopo aver ascoltato e verificato il campione scelto
assert REFERENCE_CHECKED, "Ascolta il campione e verifica che corrisponda al testo, poi imposta REFERENCE_CHECKED = True."
assert 1 <= SELECTED_CANDIDATE <= len(CANDIDATE_SEEDS)
REFERENCE_PATH = WORK / f"reference-{SELECTED_CANDIDATE}.wav"
assert REFERENCE_PATH.exists(), "Genera prima i campioni."
reference_metadata = json.loads(REFERENCE_PATH.with_suffix(".json").read_text(encoding="utf-8"))
reference_hash = hashlib.sha256(REFERENCE_PATH.read_bytes()).hexdigest()
OUTPUT = WORK / f"audio-{reference_hash[:10]}-{BASE_MODEL.split('/')[-1]}"
OUTPUT.mkdir(parents=True, exist_ok=True)
print("Riferimento scelto:", REFERENCE_PATH.name)


## 5 · Testi dei 24 audio
Puoi correggere un testo prima della generazione. I nomi dei 18 passaggi corrispondono agli identificativi dell’app. Questi testi sono una fotografia del prototipo al 15 novembre 2026.


In [ ]:
AUDIO_SCRIPTS = [{'id': 'preparation', 'category': 'step', 'title': 'Counseling prenatale · Briefing del team · Checklist del materiale', 'text': 'Counseling prenatale. Briefing del team. Checklist del materiale. Partecipa al counseling prenatale e al briefing; chiarisci il compito assegnato e verifica il materiale prima della nascita. Interventi: Counseling prenatale. Briefing del team. Checklist del materiale. Al termine, premi Prosegui.', 'source': 'Poster · Counseling prenatale / briefing / checklist'}, {'id': 'birth', 'category': 'step', 'title': 'Nascita', 'text': 'Nascita. Registra quando è avvenuta la nascita: questo è il riferimento per il tempo trascorso e i target saturazione periferica di ossigeno. Inserisci i minuti trascorsi dalla nascita, poi premi Registra nascita e prosegui.', 'source': 'Poster · Nascita'}, {'id': 'cord', 'category': 'step', 'title': 'Gestione cordone ombelicale', 'text': 'Gestione cordone ombelicale. Concorda il clampaggio con il team e assisti la gestione del cordone. Non ritardare gli interventi necessari. Interventi: Gestione cordone ombelicale. Clampaggio e milking: attendere almeno 60 secondi prima di clampare; considerare milking nei neonati depressi tra 35 e 42 settimane. Non ritardare gli interventi necessari: concorda la gestione con il team. Al termine, premi Prosegui.', 'source': 'Poster · Gestione cordone ombelicale'}, {'id': 'initial', 'category': 'step', 'title': 'Gravidanza a termine? Buon tono? Respira o piange?', 'text': 'Gravidanza a termine? Buon tono? Respira o piange?. Verifica l’età gestazionale, valuta il tono e osserva se il neonato respira o piange. Registra separatamente le tre risposte. Indica separatamente: gravidanza a termine, buon tono, respira o piange. Poi conferma la valutazione.', 'source': 'Poster · Primo bivio'}, {'id': 'skin', 'category': 'step', 'title': 'Skin to skin · Controllo della temperatura · Valutazione continua', 'text': 'Skin to skin. Controllo della temperatura. Valutazione continua. Assisti il contatto skin to skin, controlla la temperatura e continua a valutare il neonato. La valutazione prosegue nel tempo; non si esaurisce con una conferma. Interventi: Skin to skin. Controllo della temperatura. Valutazione continua', 'source': 'Poster · Skin to skin / controllo temperatura / valutazione continua'}, {'id': 'first-actions', 'category': 'step', 'title': 'Fornire calore e mantenere la temperatura · Asciugare · Posizione idonea per la pervietà delle vie aeree · Rimuovere le secrezioni se necessario · Stimolare · Riposizionare', 'text': 'Fornire calore e mantenere la temperatura. Asciugare. Posizione idonea per la pervietà delle vie aeree. Rimuovere le secrezioni se necessario. Stimolare. Riposizionare. Mantieni la temperatura, asciuga e posiziona il neonato per mantenere pervie le vie aeree. Rimuovi le secrezioni solo se necessario, stimola e riposiziona; poi valuta apnea o gasping e frequenza cardiaca. Interventi: Fornire calore e mantenere la temperatura. Asciugare. Posizione idonea per la pervietà delle vie aeree. Rimuovere le secrezioni se necessario. Stimolare. Riposizionare. Al termine, premi Prosegui.', 'source': 'Poster · Primi passi'}, {'id': 'respiration', 'category': 'step', 'title': 'Apnea e/o gasping? FC <100/min?', 'text': 'Apnea o gasping? frequenza cardiaca inferiore a 100 battiti al minuto?. Osserva se sono presenti apnea o gasping e rileva la frequenza cardiaca. Apnea/gasping o frequenza cardiaca inferiore a 100 battiti al minuto richiedono supporto ventilatorio. Inserisci la frequenza cardiaca rilevata in battiti al minuto e indica se sono presenti apnea o gasping. Poi conferma la valutazione.', 'source': 'Poster · Bivio apnea/gasping o FC <100/min'}, {'id': 'distress', 'category': 'step', 'title': 'Respiro difficoltoso e/o cianosi persistente?', 'text': 'Respiro difficoltoso o cianosi persistente?. Osserva il respiro e verifica la presenza di cianosi persistente. Se è presente almeno una delle due condizioni, seleziona SÌ. Indica se sono presenti respiro difficoltoso o cianosi persistente. Poi conferma la valutazione.', 'source': 'Poster · Bivio respiro difficoltoso / cianosi persistente'}, {'id': 'distress-support', 'category': 'step', 'title': 'Posizione idonea per la pervietà delle vie aeree · Monitoraggio SpO₂ · Supplementazione O₂ · Considerare CPAP', 'text': 'Posizione idonea per la pervietà delle vie aeree. Monitoraggio saturazione periferica di ossigeno. Supplementazione ossigeno. Considerare pressione positiva continua nelle vie aeree. Mantieni pervie le vie aeree e attiva il monitoraggio saturazione periferica di ossigeno. Collabora alla supplementazione di ossigeno e valuta con il team la necessità di pressione positiva continua nelle vie aeree. Interventi: Posizione idonea per la pervietà delle vie aeree. Monitoraggio saturazione periferica di ossigeno. Supplementazione ossigeno. Considerare pressione positiva continua nelle vie aeree. Al termine, premi Prosegui.', 'source': 'Poster · Ramo respiro difficoltoso'}, {'id': 'ppv', 'category': 'step', 'title': 'PPV · Monitoraggio SpO₂ · Considerare monitoraggio ECG', 'text': 'ventilazione a pressione positiva. Monitoraggio saturazione periferica di ossigeno. Considerare monitoraggio elettrocardiogramma. Esegui o assisti la ventilazione a pressione positiva. Monitora la saturazione periferica di ossigeno, valuta con il team il monitoraggio elettrocardiogramma e rivaluta la frequenza cardiaca. Interventi: ventilazione a pressione positiva. Monitoraggio saturazione periferica di ossigeno. Considerare monitoraggio elettrocardiogramma. Al termine, premi Prosegui.', 'source': 'Poster · PPV'}, {'id': 'hr100', 'category': 'step', 'title': 'FC <100/min?', 'text': 'frequenza cardiaca inferiore a 100 battiti al minuto?. Rileva la frequenza cardiaca dopo la ventilazione. Se rimane inferiore a 100 battiti al minuto, correggi la ventilazione; altrimenti prosegui con le cure post-rianimatorie. Inserisci la frequenza cardiaca rilevata in battiti al minuto, poi conferma la valutazione.', 'source': 'Poster · Bivio dopo PPV'}, {'id': 'correctives', 'category': 'step', 'title': 'Correttivi della ventilazione · Se necessario tubo tracheale o maschera laringea · Monitor ECG', 'text': 'Correttivi della ventilazione. Se necessario tubo tracheale o maschera laringea. Monitor elettrocardiogramma. Verifica maschera, posizione, secrezioni, apertura della bocca e pressioni. Se necessario prepara una via aerea alternativa e attiva il monitoraggio elettrocardiogramma. Interventi: Correttivi della ventilazione. Se necessario tubo tracheale o maschera laringea. Monitor elettrocardiogramma. Al termine, premi Prosegui.', 'source': 'Poster · Correttivi della ventilazione'}, {'id': 'hr60', 'category': 'step', 'title': 'FC <60/min?', 'text': 'frequenza cardiaca inferiore a 60 battiti al minuto?. Rileva la frequenza cardiaca dopo i correttivi. Prima di avviare le compressioni, conferma con il team che la ventilazione sia efficace. Inserisci la frequenza cardiaca rilevata in battiti al minuto. Prima delle compressioni, conferma con il team la ventilazione efficace. Poi conferma la valutazione.', 'source': 'Poster · Bivio dopo correttivi / messaggi chiave'}, {'id': 'compressions', 'category': 'step', 'title': 'O₂ al 100% · Tubo tracheale o maschera laringea · Compressioni toraciche coordinate con PPV · Monitoraggio ECG · Considerare CVO o IO', 'text': 'ossigeno al 100 per cento. Tubo tracheale o maschera laringea. Compressioni toraciche coordinate con ventilazione a pressione positiva. Monitoraggio elettrocardiogramma. Considerare catetere venoso ombelicale o accesso intraosseo. Collabora all’erogazione di ossigeno al 100 per cento e alla gestione della via aerea. Esegui o assisti le compressioni coordinate con ventilazione a pressione positiva secondo il ruolo assegnato, monitora l’elettrocardiogramma e prepara il materiale per catetere venoso ombelicale o accesso intraosseo se indicato dal team. Interventi: ossigeno al 100 per cento. Tubo tracheale o maschera laringea. Compressioni toraciche coordinate con ventilazione a pressione positiva. Monitoraggio elettrocardiogramma. Considerare catetere venoso ombelicale o accesso intraosseo. Al termine, premi Prosegui.', 'source': 'Poster · Compressioni toraciche coordinate con PPV'}, {'id': 'advanced-hr', 'category': 'step', 'title': 'FC <60/min?', 'text': 'frequenza cardiaca inferiore a 60 battiti al minuto?. Rileva e comunica la frequenza cardiaca dopo le compressioni coordinate con ventilazione a pressione positiva. Registra il valore per verificare se rimane inferiore a 60 battiti al minuto. Inserisci la frequenza cardiaca rilevata in battiti al minuto, poi conferma la valutazione.', 'source': 'Poster · Bivio dopo compressioni'}, {'id': 'medication', 'category': 'step', 'title': 'Adrenalina in CVO o IO ogni 3–5 min · Se FC <60/min persistente considerare ipovolemia · DOPPES', 'text': 'Adrenalina in catetere venoso ombelicale o accesso intraosseo ogni 3 a 5 minuti. Se frequenza cardiaca inferiore a 60 battiti al minuto persistente considerare ipovolemia. Doppes. Verifica la prescrizione e prepara l’adrenalina per la via catetere venoso ombelicale o accesso intraosseo. Registra ogni somministrazione e controlla il tempo trascorso; se frequenza cardiaca rimane inferiore a 60 battiti al minuto, collabora alla valutazione di ipovolemia e delle cause di ventilazione inefficace. Interventi: Adrenalina in catetere venoso ombelicale o accesso intraosseo ogni 3 a 5 minuti. Se frequenza cardiaca inferiore a 60 battiti al minuto persistente, considerare ipovolemia. Rivalutazione Doppes. Al termine, premi Prosegui.', 'source': 'Poster · Adrenalina / ipovolemia / DOPPES'}, {'id': 'recovery-review', 'category': 'step', 'title': 'Rivalutare la risposta del neonato', 'text': 'Rivalutare la risposta del neonato. Comunica la frequenza cardiaca rilevata, rivaluta il respiro e concorda con il team la prosecuzione del supporto. Interventi: Risposta del neonato rivalutata con il team', 'source': 'Limite esplicito della trasposizione del diagramma'}, {'id': 'post', 'category': 'step', 'title': 'Cure post-rianimatorie · Debriefing', 'text': 'Cure post-rianimatorie. Debriefing. Prosegui l’assistenza post-rianimatoria secondo le indicazioni del team e partecipa al debriefing, riferendo valutazioni, interventi e tempi registrati. Interventi: Cure post-rianimatorie. Debriefing del team', 'source': 'Poster · Cure post-rianimatorie / debriefing'}, {'id': 'saturation', 'category': 'reference', 'title': 'Target di saturazione preduttale', 'text': 'Target di saturazione preduttale. A 2 minuti: dal 65 al 70 per cento. A 3 minuti: dal 70 al 75 per cento. A 4 minuti: dal 75 all’80 per cento. A 5 minuti: dall’80 all’85 per cento. A 10 minuti: dall’85 al 95 per cento.', 'source': 'Contenuti del Nido'}, {'id': 'mrsopa', 'category': 'reference', 'title': 'Correttivi MR SOPA', 'text': 'Correttivi della ventilazione in maschera. Maschera adeguata al volto. Riposizionamento in sniffing position. Verifica secrezioni e aspira le vie aeree. Apri (Open) la bocca. Pressioni adeguate. Via aerea alternativa.', 'source': 'Contenuti del Nido'}, {'id': 'doppes', 'category': 'reference', 'title': 'Verifiche DOPPES', 'text': 'Cause di ventilazione inefficace nel neonato intubato. Dislocazione. Ostruzione. Pressioni adeguate. Pneumotorace. Strumentazione (Equipment) ben funzionante. Rimuovere aria dallo stomaco.', 'source': 'Contenuti del Nido'}, {'id': 'newborn', 'category': 'interface', 'title': 'Dati dichiarati nello scenario', 'text': 'Inserisci, se disponibili, l’età gestazionale in settimane compiute e la temperatura rilevata in gradi Celsius. Poi premi Registra dati dello scenario.', 'source': 'Contenuti del Nido'}, {'id': 'help', 'category': 'interface', 'title': 'Aiuto e coordinamento del team', 'text': 'Attiva il sistema di chiamata previsto dalla struttura e condividi il passaggio attuale. Nessuna chiamata viene effettuata dall’app. Conferma sullo schermo la richiesta effettuata nello scenario.', 'source': 'Contenuti del Nido'}, {'id': 'finished', 'category': 'interface', 'title': 'Simulazione conclusa', 'text': 'Simulazione conclusa. Puoi esportare il registro o avviare un’altra sessione. Il registro corrente viene perso chiudendo o ricaricando la pagina.', 'source': 'Contenuti del Nido'}]
assert len(AUDIO_SCRIPTS) == 24
print("Testi caricati:", len(AUDIO_SCRIPTS))


## 6 · Carica il modello Base e prepara la voce
Il modello VoiceDesign è già stato liberato. Se la memoria non basta, cambia `BASE_MODEL` nella configurazione in `Qwen/Qwen3-TTS-12Hz-0.6B-Base`, riesegui la selezione e riparti da questa cella.


In [ ]:
release_models()
clone_model = load_model(BASE_MODEL)
with torch.inference_mode():
    voice_prompt = clone_model.create_voice_clone_prompt(
        ref_audio=str(REFERENCE_PATH), ref_text=reference_metadata["text"],
        x_vector_only_mode=False)
print("Voce pronta per la generazione.")


## 7 · Genera tutti gli audio
Un testo alla volta, suddiviso in frasi per contenere memoria e lunghezza. Le frasi vengono unite con brevi pause. I segmenti validi vengono salvati: rieseguire riprende il lavoro nella stessa sessione. La verifica automatica controlla i file, non la correttezza delle parole pronunciate.


In [ ]:
CACHE = OUTPUT / "segments"
CACHE.mkdir(exist_ok=True)
settings = {"model": BASE_MODEL, "reference_sha256": reference_hash,
    "qwen_tts_version": importlib.metadata.version("qwen-tts"),
    "precision": str(DTYPE), "attention": ATTENTION, "language": "Italian",
    "max_new_tokens": 2048, "do_sample": False, "pause_seconds": 0.18}
manifest = {"settings": settings, "reference": reference_metadata,
    "purpose": "Simulazione; pronunce e contenuti da verificare mediante ascolto", "clips": []}
failures = []

def split_text(text):
    sentences = re.split(r"(?<=[.!?;])\s+", text.strip())
    return [sentence.strip() for sentence in sentences if sentence.strip()]

def save_manifest():
    (OUTPUT / "manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
    (OUTPUT / "errors.json").write_text(json.dumps(failures, ensure_ascii=False, indent=2), encoding="utf-8")

for index, clip in enumerate(AUDIO_SCRIPTS, 1):
    try:
        pieces = []
        output_sr = None
        for part_number, text in enumerate(split_text(clip["text"]), 1):
            fingerprint = hashlib.sha256(json.dumps({"text": text, **settings},
                sort_keys=True, ensure_ascii=False).encode()).hexdigest()
            segment_path = CACHE / f"{fingerprint}.wav"
            if segment_path.exists():
                audio, sr = sf.read(segment_path, dtype="float32")
                audio = checked_audio(audio)
            else:
                with torch.inference_mode():
                    waves, sr = clone_model.generate_voice_clone(
                        text=text, language="Italian", voice_clone_prompt=voice_prompt,
                        max_new_tokens=settings["max_new_tokens"], do_sample=False)
                audio = checked_audio(waves[0])
                # A very long output for a short phrase may indicate looping.
                if len(audio) / sr > max(45, len(text) * 0.25):
                    raise ValueError(f"Segmento troppo lungo, possibile ripetizione: {text}")
                sf.write(segment_path, audio, int(sr), subtype="PCM_16")
            if output_sr is None:
                output_sr = int(sr)
            if int(sr) != output_sr:
                raise ValueError("Frequenze di campionamento diverse nello stesso audio.")
            if pieces:
                pieces.append(np.zeros(round(output_sr * settings["pause_seconds"]), dtype=np.float32))
            pieces.append(audio)
        if not pieces:
            raise ValueError("Testo vuoto.")
        combined = checked_audio(np.concatenate(pieces))
        destination = OUTPUT / f"{clip['id']}.wav"
        sf.write(destination, combined, output_sr, subtype="PCM_16")
        (OUTPUT / f"{clip['id']}.txt").write_text(clip["text"], encoding="utf-8")
        # Reopen the actual saved file before listing it in the manifest.
        info = sf.info(destination)
        assert info.frames > 0 and info.samplerate == output_sr
        manifest["clips"].append({**clip, "file": destination.name,
            "sample_rate": output_sr, "duration_seconds": round(info.duration, 2),
            "sha256_audio": hashlib.sha256(destination.read_bytes()).hexdigest()})
        print(f"[{index}/{len(AUDIO_SCRIPTS)}] {destination.name}: {info.duration:.1f} s")
    except Exception as exc:
        failures.append({"id": clip["id"], "error": str(exc)})
        print(f"ERRORE {clip['id']}: {exc}")
        gc.collect()
        torch.cuda.empty_cache()
    save_manifest()

print(f"Completati: {len(manifest['clips'])}/{len(AUDIO_SCRIPTS)}. Errori: {len(failures)}.")
if failures:
    print("Controlla errors.json. Dopo la correzione riesegui questa cella: i segmenti completati saranno riutilizzati.")
else:
    display(Audio(filename=str(OUTPUT / "respiration.wav")))


## 8 · ZIP e download
Lo ZIP completo viene creato solo quando tutti i 24 audio sono disponibili. Include WAV, testi, manifest, versione delle librerie e voce di riferimento; esclude la cache dei segmenti. L’opzione di recupero permette di scaricare anche un lavoro incompleto, con nome esplicito.


In [ ]:
ALLOW_PARTIAL_DOWNLOAD = False
assert "manifest" in globals(), "Esegui prima la generazione."
complete = len(manifest["clips"]) == len(AUDIO_SCRIPTS) and not failures
assert complete or ALLOW_PARTIAL_DOWNLOAD, "Generazione incompleta. Correggi gli errori o abilita il download parziale."
save_manifest()
shutil.copyfile(REFERENCE_PATH, OUTPUT / "voice-reference.wav")
(OUTPUT / "voice-reference.json").write_text(json.dumps(reference_metadata, ensure_ascii=False, indent=2), encoding="utf-8")
(OUTPUT / "README.txt").write_text(
    "Audio generati con Qwen3-TTS. Contenuti per simulazione, da verificare mediante ascolto.\n"
    "I file non sono stati collegati automaticamente all'app.\n"
    "La voce di riferimento e il manifest permettono di identificare le impostazioni usate.\n"
    "Modelli e codice Qwen: Apache 2.0. Fonte: https://github.com/QwenLM/Qwen3-TTS\n", encoding="utf-8")
archive = WORK / ("nls-qwen-completo.zip" if complete else "nls-qwen-PARZIALE.zip")
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as bundle:
    for path in sorted(OUTPUT.iterdir()):
        if path.is_file():
            bundle.write(path, arcname=path.name)
with zipfile.ZipFile(archive) as bundle:
    assert bundle.testzip() is None
    wav_names = {name for name in bundle.namelist() if name.endswith(".wav") and name != "voice-reference.wav"}
    assert wav_names == {c["file"] for c in manifest["clips"]}
print(f"ZIP: {archive.name} | {len(manifest['clips'])} audio | {archive.stat().st_size/2**20:.1f} MiB")
from google.colab import files
files.download(str(archive))
